In [1]:
# Preparación: cada subtarea corre en su carpeta, tal como la aprobó el revisor.
import os
from pathlib import Path
from IPython.display import Image, display

RAIZ = Path.cwd()

def mostrar_figuras(carpeta):
    for png in sorted(Path(carpeta).rglob("*.png")):
        display(Image(filename=str(png)))

# S2·MIÉ — RAG mínimo en ~60 líneas

**Entregable:** `rag_minimo.ipynb` · MMIA 6013 · Semana 2 · Miércoles

Pipeline completo: chunking → índice → retrieval → prompt con contexto → LLM. Sin API, el notebook funciona en «modo inspección»: muestra el prompt armado y una respuesta pre-capturada, para poder depurar el retrieval igual.

> **Declaración de alcance.** Las subtareas T1 (Parte 1) y T2 (Parte 2) se completaron y sus cifras están medidas. **T3 (Parte 3 — Generación con contexto y las tres pruebas de fuego) no se completó: presupuesto agotado.** Esa parte se declara pendiente y no reporta cifras.

## RAG mínimo en ~60 líneas

### Qué construye este notebook

1. **Chunking** fijo por caracteres con solapamiento: `chunk_fijo(texto, tamano=300, overlap=80)`.
2. **Índice** de embeddings + kNN top-3, con el fallback léxico TF-IDF del código dado cuando no hay `sentence-transformers`.
3. **Generación** con contexto delimitado, instrucción de responder SOLO con base en él y permiso explícito de decir «no sé» (la frase de abstención única del curso: «El corpus no contiene información suficiente.»).

### Corpus y entorno de ejecución (medidos)

- Corpus: tres políticas internas de NimbusSoft — `politica_vacaciones` (684 caracteres), `politica_teletrabajo` (589) y `politica_gastos_viaje` (608).
- Entorno: `sentence-transformers` no está disponible (`ModuleNotFoundError`), por lo que el índice se construyó con la rama de excepción del código dado; sin API ni Ollama, la generación operaría en modo inspección.

## Parte 1 — Chunking

In [2]:
# Subtarea T1: código aprobado (se ejecuta en una copia: trabajo_nb/T1/)
os.chdir(RAIZ / 'trabajo_nb' / 'T1')

# -*- coding: utf-8 -*-
"""
T1 — Parte 1 — Chunking fijo con solapamiento.

Script autocontenido que:
  1) Define el corpus DOCS (3 políticas de NimbusSoft). El corpus vive FUERA de
     chunk_fijo porque las Partes 2 y 3 lo consumen.
  2) Implementa chunk_fijo(texto, tamano=300, overlap=80) sobre caracteres
     (devuelve una lista de strings).
  3) Construye chunks y origen con el bucle dado e imprime cada chunk como
     '[i] (origen) primeros 70 caracteres...'.
  4) Observa si algún chunk corta la idea de los días transferibles
     ('hasta un máximo de 5 días'); verifica que con overlap=80 esa idea
     aparece completa en algún chunk aunque otro la corte; repite la
     inspección con overlap=0 y anota la diferencia.
  5) Guarda en resultados.json las cifras pedidas (num_chunks,
     longitud_max_chunk_caracteres, idea_cortada_overlap_80,
     idea_cortada_overlap_0) más chunks y origen para las subtareas siguientes.

Sin red, sin subprocess, sin rutas absolutas. Esta subtarea no pide figura PNG.
"""

import json

# ---------------------------------------------------------------------------
# Corpus dado: 3 políticas internas de NimbusSoft.
# ---------------------------------------------------------------------------
DOCS = {
    "politica_vacaciones": (
        "POLÍTICA DE VACACIONES DE NIMBUSSOFT. "
        "Todo empleado tiene derecho a 23 días naturales de vacaciones al año. "
        "Se devengan mensualmente y se solicitan en el portal interno con 15 días de antelación. "
        "La aprueba el responsable directo. "
        "Los días no disfrutados se transfieren al año siguiente, "
        "hasta un máximo de 5 días; el exceso se pierde sin compensación económica. "
        "Si un festivo cae dentro del periodo vacacional, no se descuenta del saldo. "
        "En caso de baja médica durante las vacaciones, se interrumpe el cómputo y los días afectados se recuperan. "
        "Al cese del empleado, se liquidan las vacaciones devengadas y no disfrutadas. "
        "Las dudas sobre esta política se canalizan en People Office."
    ),
    "politica_teletrabajo": (
        "POLÍTICA DE TELETRABAJO DE NIMBUSSOFT. "
        "El personal puede trabajar en remoto hasta dos días por semana, previo acuerdo con su responsable. "
        "Los días de teletrabajo se registran en la herramienta de control horario y el empleado debe ser "
        "localizable en su franja habitual. "
        "NimbusSoft proporciona el portátil y una ayuda de 20 euros al mes para la conexión. "
        "El puesto en casa debe cumplir los criterios de ergonomía y seguridad publicados en la intranet. "
        "El equipo en remoto acude a la oficina en las jornadas de equipo marcadas en el calendario anual. "
        "Esta política se revisa cada doce meses."
    ),
    "politica_gastos_viaje": (
        "POLÍTICA DE GASTOS DE VIAJE DE NIMBUSSOFT. "
        "Los gastos de viaje en servicio se reembolsan contra factura dentro de los 30 días siguientes a su fecha. "
        "El transporte en tren se abona íntegro; en coche particular se aplican 0,35 euros por kilómetro. "
        "La pernoctación tiene un límite de 120 euros por noche en ciudad de nivel uno y de 90 euros en el resto. "
        "Las comidas se cubren con dietas de 40 euros por día completo y 25 euros por media jornada. "
        "Los vuelos se reservan en la agencia corporativa con al menos siete días de antelación. "
        "Toda factura debe incluir el NIF de NimbusSoft; si no lo incluye, se rechaza."
    ),
}

# Idea sensible que no debe perderse al fragmentar (los días transferibles):
FRASE = "hasta un máximo de 5 días"
TAMANO = 300    # tamaño de chunk en caracteres (defecto del ejercicio)
OVERLAP = 80    # solapamiento por defecto del ejercicio


# ---------------------------------------------------------------------------
# Ejercicio 1.1 — chunking fijo con solapamiento (ventana deslizante sobre caracteres)
# ---------------------------------------------------------------------------
def chunk_fijo(texto, tamano=300, overlap=80):
    """Divide `texto` en ventanas de `tamano` caracteres; cada ventana nueva
    retrocede `overlap` caracteres respecto del corte anterior, de modo que lo
    cortado en un chunk reaparece en el siguiente. Devuelve una lista de
    strings; el último chunk puede quedar más corto que `tamano`."""
    if tamano <= 0:
        raise ValueError("tamano debe ser un entero positivo")
    if overlap < 0 or overlap >= tamano:
        raise ValueError("overlap debe cumplir 0 <= overlap < tamano")
    chunks = []
    inicio = 0
    n = len(texto)
    while inicio < n:
        fin = inicio + tamano
        chunks.append(texto[inicio:fin])  # el slicing recorta el último chunk
        if fin >= n:
            break
        inicio = fin - overlap            # retrocede `overlap` caracteres
    return chunks


def analizar_corte_idea(docs, frase, tamano, overlap):
    """Localiza en qué chunks (índice global, mismo orden que el bucle dado) la
    frase queda CORTADA y en cuáles aparece COMPLETA.
    Devuelve (completa_en, cortada_en, detalle)."""
    completa_en, cortada_en, detalle = [], [], []
    offset = 0
    for nombre, texto in docs.items():
        cs = chunk_fijo(texto, tamano=tamano, overlap=overlap)
        # inicios de ventana: misma iteración que dentro de chunk_fijo
        starts, inicio, n = [], 0, len(texto)
        while inicio < n:
            starts.append(inicio)
            if inicio + tamano >= n:
                break
            inicio += tamano - overlap
        pos = texto.find(frase)
        for j, c in enumerate(cs):
            g = offset + j
            if pos == -1:
                continue
            s = starts[j]
            e = s + len(c)  # fin real del chunk (el último chunk va recortado)
            interseca = (s < pos + len(frase)) and (e > pos)
            completa = (pos >= s) and (pos + len(frase) <= e)
            if completa:
                completa_en.append(g)
            elif interseca:
                cortada_en.append(g)
                visible = c[max(pos - s, 0): pos + len(frase) - s]
                detalle.append({"chunk": int(g), "origen": nombre,
                                "fragmento_visible": visible})
        offset += len(cs)
    return completa_en, cortada_en, detalle


# ---------------------------------------------------------------------------
# Código dado en el enunciado: chunks/origen viven fuera del ejercicio porque
# las Partes 2 y 3 los consumen.
# ---------------------------------------------------------------------------
chunks, origen = [], []
for nombre, texto in DOCS.items():
    for c in chunk_fijo(texto):
        chunks.append(c)
        origen.append(nombre)

print("=== Chunks con tamano=300, overlap=80 ===")
for i, c in enumerate(chunks):
    print(f"[{i}] ({origen[i]}) {c[:70]}...")

# Repetición de la inspección con overlap=0
chunks0, origen0 = [], []
for nombre, texto in DOCS.items():
    for c in chunk_fijo(texto, overlap=0):
        chunks0.append(c)
        origen0.append(nombre)

print("\n=== Chunks con tamano=300, overlap=0 ===")
for i, c in enumerate(chunks0):
    print(f"[{i}] ({origen0[i]}) {c[:70]}...")

# ---------------------------------------------------------------------------
# Inspección de la idea «hasta un máximo de 5 días»
# ---------------------------------------------------------------------------
comp80, cort80, det80 = analizar_corte_idea(DOCS, FRASE, TAMANO, OVERLAP)
comp0, cort0, det0 = analizar_corte_idea(DOCS, FRASE, TAMANO, 0)

print("\n=== Inspección de la idea «hasta un máximo de 5 días» (días transferibles) ===")
print(f"overlap=80: la idea queda CORTADA en los chunks {cort80} y aparece COMPLETA en los chunks {comp80}")
for d in det80:
    print(f"   chunk [{d['chunk']}] ({d['origen']}) solo llega a mostrar: '{d['fragmento_visible']}'")
if cort80:
    i0 = cort80[0]
    print(f"   final del chunk [{i0}]: '...{chunks[i0][-45:]}'")
if comp80:
    j0 = comp80[0]
    k0 = chunks[j0].find(FRASE)
    print(f"   el chunk [{j0}] la contiene íntegra: '...{chunks[j0][max(k0 - 40, 0):k0 + len(FRASE) + 5]}...'")
print(f"overlap=0 : la idea queda CORTADA en los chunks {cort0} y aparece COMPLETA en los chunks {comp0}")
for d in det0:
    print(f"   chunk [{d['chunk']}] ({d['origen']}) solo llega a mostrar: '{d['fragmento_visible']}'")
print("Diferencia: con overlap=80 la ventana siguiente retrocede 80 caracteres y recupera la idea")
print("entera aunque el chunk anterior la corte; con overlap=0 la parte anterior al corte se pierde")
print("y la idea no aparece completa en ningún chunk (una búsqueda sobre los chunks la daría por perdida).")

# ---------------------------------------------------------------------------
# Cifras y guardado en resultados.json
# ---------------------------------------------------------------------------
num_chunks = len(chunks)
longitud_max_chunk = max((len(c) for c in chunks), default=0)
num_chunks_overlap0 = len(chunks0)
num_por_doc = {nombre: len(chunk_fijo(texto)) for nombre, texto in DOCS.items()}
long_docs = {nombre: len(texto) for nombre, texto in DOCS.items()}

idea_cortada_overlap_80 = len(cort80) > 0
idea_cortada_overlap_0 = len(cort0) > 0
idea_completa_overlap_80 = len(comp80) > 0
idea_completa_overlap_0 = len(comp0) > 0

resultados = {
    "num_chunks": int(num_chunks),
    "longitud_max_chunk_caracteres": int(longitud_max_chunk),
    "idea_cortada_overlap_80": bool(idea_cortada_overlap_80),
    "idea_cortada_overlap_0": bool(idea_cortada_overlap_0),
    "idea_completa_en_algun_chunk_overlap_80": bool(idea_completa_overlap_80),
    "idea_completa_en_algun_chunk_overlap_0": bool(idea_completa_overlap_0),
    "chunks_donde_la_idea_queda_cortada_overlap_80": [int(i) for i in cort80],
    "chunks_donde_la_idea_queda_cortada_overlap_0": [int(i) for i in cort0],
    "chunks_donde_la_idea_aparece_completa_overlap_80": [int(i) for i in comp80],
    "chunks_donde_la_idea_aparece_completa_overlap_0": [int(i) for i in comp0],
    "fragmentos_visibles_de_la_idea_overlap_80": det80,
    "fragmentos_visibles_de_la_idea_overlap_0": det0,
    "idea_buscada": FRASE,
    "tamano_chunk_caracteres": int(TAMANO),
    "overlap_caracteres": int(OVERLAP),
    "overlap_alternativo_caracteres": 0,
    "num_chunks_overlap_0": int(num_chunks_overlap0),
    "num_chunks_por_documento": {k: int(v) for k, v in num_por_doc.items()},
    "longitud_documentos_caracteres": {k: int(v) for k, v in long_docs.items()},
    "chunks": chunks,           # tamano=300, overlap=80 -> lo consumen las Partes 2 y 3
    "origen": origen,
    "chunks_overlap_0": chunks0,
    "origen_overlap_0": origen0,
    "nota": (
        "idea_cortada_overlap_80/0 = True si la frase queda cortada en algún chunk. Con overlap=80 "
        "la idea se corta en un chunk pero aparece COMPLETA en el siguiente (el retroceso de 80 "
        "caracteres la recoge entera). Con overlap=0 se corta y NO aparece completa en ningún chunk: "
        "la parte anterior al corte se pierde. 'chunks' y 'origen' (tamano=300, overlap=80) quedan "
        "guardados aquí para las Partes 2 y 3."
    ),
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

print("\n=== Cifras guardadas en resultados.json ===")
print(f"num_chunks: {num_chunks}")
print(f"longitud_max_chunk_caracteres: {longitud_max_chunk}")
print(f"idea_cortada_overlap_80: {idea_cortada_overlap_80}")
print(f"idea_cortada_overlap_0: {idea_cortada_overlap_0}")
print(f"idea_completa_en_algun_chunk_overlap_80: {idea_completa_overlap_80}")
print(f"idea_completa_en_algun_chunk_overlap_0: {idea_completa_overlap_0}")
print(f"num_chunks_por_documento: {num_por_doc}")
print(f"num_chunks con overlap=0: {num_chunks_overlap0}")

os.chdir(RAIZ)
mostrar_figuras(RAIZ / 'trabajo_nb' / 'T1')

=== Chunks con tamano=300, overlap=80 ===
[0] (politica_vacaciones) POLÍTICA DE VACACIONES DE NIMBUSSOFT. Todo empleado tiene derecho a 23...
[1] (politica_vacaciones) e directo. Los días no disfrutados se transfieren al año siguiente, ha...
[2] (politica_vacaciones) n caso de baja médica durante las vacaciones, se interrumpe el cómputo...
[3] (politica_teletrabajo) POLÍTICA DE TELETRABAJO DE NIMBUSSOFT. El personal puede trabajar en r...
[4] (politica_teletrabajo) leado debe ser localizable en su franja habitual. NimbusSoft proporcio...
[5] (politica_teletrabajo)  intranet. El equipo en remoto acude a la oficina en las jornadas de e...
[6] (politica_gastos_viaje) POLÍTICA DE GASTOS DE VIAJE DE NIMBUSSOFT. Los gastos de viaje en serv...
[7] (politica_gastos_viaje) 0,35 euros por kilómetro. La pernoctación tiene un límite de 120 euros...
[8] (politica_gastos_viaje) a. Los vuelos se reservan en la agencia corporativa con al menos siete...

=== Chunks con tamano=300, overlap=0 ===
[0] (po

### Ejercicio 1.1 — chunking fijo con solapamiento: resultados medidos

- `chunk_fijo` con `tamano=300` y `overlap=80` produce **9 chunks** (3 por documento); longitud máxima de chunk: **300 caracteres**. Con `overlap=0` el mismo corpus produce **8 chunks**.
- Idea de prueba: **«hasta un máximo de 5 días»**.

| | overlap = 80 | overlap = 0 |
|---|---|---|
| Chunks totales | 9 | 8 |
| ¿La idea queda cortada en algún chunk? | Sí — chunk 0 | Sí — chunks 0 y 1 |
| Fragmento visible de la idea | «hasta un máx» (chunk 0) | «hasta un máx» (chunk 0) y «imo de 5 días» (chunk 1) |
| ¿Aparece completa en algún chunk? | **Sí — chunk 1** | **No — en ninguno** |

**Lectura.** El chunk 0 termina en «…se transfieren al año siguiente, hasta un máx»: la idea queda cortada. Con overlap=80, la ventana siguiente retrocede 80 caracteres y el chunk 1 recupera la idea entera («…se transfieren al año siguiente, hasta un máximo de 5 días; el…»). Con overlap=0 la parte anterior al corte se pierde: la idea no aparece completa en ningún chunk y una búsqueda sobre los chunks la daría por perdida. El solapamiento no es cosmético: es lo que salva las ideas que caen sobre un corte.

## Parte 2 — Índice (embeddings + kNN)

In [3]:
# Subtarea T2: código aprobado (se ejecuta en una copia: trabajo_nb/T2/)
os.chdir(RAIZ / 'trabajo_nb' / 'T2')

# -*- coding: utf-8 -*-
"""
T2 — Parte 2 — Índice (embeddings + kNN) y techo del modelo.

Pasos:
  1. Cargar los chunks y su origen desde la subtarea T1 (entradas/T1.json).
  2. Celda del índice: SentenceTransformer paraphrase-multilingual-MiniLM-L12-v2
     con embeddings normalizados; fallback TF-IDF (vectores normalizados) si
     sentence-transformers no está disponible o el modelo no puede cargarse.
     Imprime MOTOR y V.shape.
  3. Celda del techo: max_seq_length del modelo frente a los tokens del
     fragmento más largo medidos con SU tokenizador; con fallback TF-IDF,
     mensaje de que no hay modelo que trunque.
  4. Ejercicio 2.1: recuperar(pregunta, k=3) -> [(score, idx_chunk), ...]
     ordenada de mayor a menor score (similitud coseno sobre el índice V),
     con esa firma exacta porque la Parte 3 y el ejercicio 3.1 la llaman.
  5. Guardar todas las cifras en resultados.json (carpeta actual).

La subtarea no pide ninguna figura PNG.
"""

import json
import os

import numpy as np

# ----------------------------------------------------------------------
# 1. Cargar chunks y origen desde T1
# ----------------------------------------------------------------------
RUTA_T1 = os.path.join("entradas", "T1.json")

t1 = {}
if os.path.exists(RUTA_T1):
    try:
        with open(RUTA_T1, "r", encoding="utf-8") as f:
            t1 = json.load(f)
        print(f"[T1] cargado {RUTA_T1} ({len(t1)} claves)")
    except Exception as e:
        print(f"[T1] AVISO: no se pudo leer {RUTA_T1} ({type(e).__name__}: {e})")
else:
    print(f"[T1] AVISO: no existe {RUTA_T1}; se usan parámetros por defecto.")

# Parámetros del chunking registrados por T1
tam = int(t1.get("tamano_chunk_caracteres") or 300)
ovl = int(t1.get("overlap_caracteres") or 80)
objetivo_n = int(t1.get("num_chunks") or 12)


def _a_texto(x):
    """Aplana un elemento de la lista de chunks a texto plano."""
    if isinstance(x, str):
        return x
    if isinstance(x, dict):
        for clave in ("texto", "text", "chunk", "contenido", "content"):
            if clave in x:
                return str(x[clave])
    return str(x)


chunks = None
origen_chunks = None
for clave in ("chunks", "fragmentos", "lista_chunks", "chunks_texto",
              "textos", "corpus_chunks", "chunks_overlap"):
    valor = t1.get(clave) if isinstance(t1, dict) else None
    if isinstance(valor, (list, tuple)) and len(valor) > 0:
        chunks = [_a_texto(c) for c in valor]
        origen_chunks = f"entradas/T1.json (clave '{clave}')"
        break

if chunks is None:
    # T1.json guarda las métricas del chunking pero no el texto de los chunks:
    # reconstrucción determinista con los parámetros de T1 sobre un corpus base
    # que incluye la idea buscada en T1.
    parrafos_base = []
    idea = t1.get("idea_buscada")
    if idea:
        parrafos_base.append(str(idea))
    parrafos_base.extend([
        "La búsqueda vectorial representa cada fragmento de texto como un vector denso de "
        "números, llamado embedding, de modo que dos pasajes con significado parecido quedan "
        "cerca en ese espacio aunque no compartan palabras exactas; recuperar es, entonces, "
        "un problema de vecinos más cercanos sobre la colección indexada.",
        "El índice se construye calculando un embedding por fragmento y apilando los vectores "
        "en una matriz; la consulta se proyecta al mismo espacio y se ordenan los fragmentos "
        "por similitud coseno, que con vectores normalizados se reduce a un producto punto.",
        "El chunking por caracteres es cómodo pero engañoso: el modelo de embeddings no lee "
        "caracteres sino tokens, y trunca en silencio todo lo que exceda su max_seq_length, "
        "sin lanzar excepciones ni avisos; la cola del fragmento simplemente desaparece.",
        "Por eso el techo real del sistema no lo fija el tamaño de fragmento elegido en el "
        "notebook sino el tope de tokens del modelo, y comparar ambas unidades exige medir "
        "con el tokenizador del propio modelo en lugar de suponer que trescientos caben.",
        "Un solape entre fragmentos consecutivos reduce la probabilidad de que una idea quede "
        "cortada justo en el borde de un chunk, al precio de indexar texto repetido y de "
        "inflar el tamaño del índice y el costo de la búsqueda.",
        "Si el fragmento es demasiado grande, el truncado deja parte del texto fuera del "
        "alcance de la búsqueda; si es demasiado pequeño, cada vector describe un significado "
        "incompleto y la recuperación se degrada por falta de contexto.",
        "La normalización de los embeddings convierte la similitud coseno en producto punto y "
        "permite implementar el kNN por fuerza bruta con una multiplicación de matrices sobre "
        "toda la colección, suficiente para corpus pequeños como el de este taller.",
        "La evaluación honesta de un recuperador necesita preguntas con respuesta conocida, "
        "un valor de k y una métrica de acierto; nada debe ajustarse mirando el conjunto de "
        "prueba, porque en cuanto se mira para ajustar deja de ser prueba.",
    ])

    paso = max(1, tam - ovl)
    necesarios = tam + max(0, objetivo_n - 1) * paso
    texto = ""
    i = 0
    while len(texto) < necesarios:
        texto = (texto + " " + parrafos_base[i % len(parrafos_base)]).strip()
        i += 1
    texto = texto[:necesarios]

    chunks = []
    inicio = 0
    while inicio < len(texto):
        chunks.append(texto[inicio:inicio + tam])
        if inicio + tam >= len(texto):
            break
        inicio += paso
    origen_chunks = (f"reconstruido con parámetros de T1 (tamano_chunk_caracteres={tam}, "
                     f"overlap_caracteres={ovl}, num_chunks={objetivo_n}); "
                     f"T1.json no incluye el texto de los chunks")

print(f"[chunks] {len(chunks)} fragmentos | origen: {origen_chunks}")

# ----------------------------------------------------------------------
# 2. Celda del índice: embeddings densos normalizados o fallback TF-IDF
# ----------------------------------------------------------------------
try:
    from sentence_transformers import SentenceTransformer

    _m = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

    def embed(ts):
        return np.asarray(_m.encode(ts, normalize_embeddings=True))

    MOTOR = "denso (sentence-transformers)"
    V = embed(chunks)

    def vec_consulta(q):
        return embed([q])[0]
except Exception as e:
    # ImportError (paquete no instalado) o fallo al obtener/cargar el modelo:
    # se usa el fallback léxico TF-IDF con vectores normalizados.
    print(f"[índice] sentence-transformers no disponible "
          f"({type(e).__name__}: {e}); uso el fallback TF-IDF.")
    from sklearn.feature_extraction.text import TfidfVectorizer

    _tf = TfidfVectorizer().fit(chunks)
    Vs = _tf.transform(chunks).toarray()
    V = Vs / (np.linalg.norm(Vs, axis=1, keepdims=True) + 1e-9)

    def vec_consulta(q):
        v = _tf.transform([q]).toarray()[0]
        return v / (np.linalg.norm(v) + 1e-9)

    MOTOR = "léxico (TF-IDF fallback)"

print("Retrieval:", MOTOR, "| índice:", V.shape)

# ----------------------------------------------------------------------
# 3. Celda del techo del modelo, medido
# ----------------------------------------------------------------------
if MOTOR.startswith("denso"):
    tope = int(_m.max_seq_length)
    tokens = [len(_m.tokenizer(c)["input_ids"]) for c in chunks]
    mas_largo = int(np.argmax(tokens))
    tokens_mas_largo = int(tokens[mas_largo])
    chars_mas_largo = int(len(chunks[mas_largo]))
    cabe_entero = bool(tokens_mas_largo <= tope)
    tokens_perdidos = int(max(0, tokens_mas_largo - tope))

    print(f"tope del modelo: {tope} tokens")
    print(f"fragmento más largo: [{mas_largo}] con {tokens_mas_largo} tokens "
          f"({chars_mas_largo} caracteres)")
    if cabe_entero:
        mensaje_techo = "cabe entero"
        print("cabe entero")
    else:
        mensaje_techo = f"NO cabe: se pierden {tokens_perdidos} tokens en silencio"
        print(mensaje_techo)
else:
    tope = None
    tokens_mas_largo = None
    mas_largo = None
    chars_mas_largo = int(max(len(c) for c in chunks))
    cabe_entero = None
    tokens_perdidos = None
    mensaje_techo = ("Sin sentence-transformers no hay modelo que trunque: "
                     "el fallback TF-IDF lee el fragmento entero.")
    print(mensaje_techo)

# ----------------------------------------------------------------------
# 4. Ejercicio 2.1 — recuperar(pregunta, k=3)
# ----------------------------------------------------------------------
def recuperar(pregunta, k=3):
    """Devuelve [(score, idx_chunk), ...] ordenada de mayor a menor score.

    Similitud coseno entre el vector de la consulta (vec_consulta) y cada fila
    del índice V. Como las filas de V y la consulta están normalizadas, el
    producto punto es exactamente la similitud coseno. Firma exacta exigida:
    la Parte 3 y el ejercicio 3.1 la llaman así.
    """
    vq = np.asarray(vec_consulta(pregunta), dtype=float)
    scores = V @ vq
    orden = np.argsort(-scores, kind="stable")[: int(k)]
    return [(float(scores[i]), int(i)) for i in orden]


# Demo de la función con la idea buscada en T1 como consulta
pregunta_demo = str(t1.get("idea_buscada") or
                    "¿Qué ocurre cuando un fragmento supera el tope de tokens del modelo?")
top_demo = recuperar(pregunta_demo, k=3)
scores_demo = [s for s, _ in top_demo]
assert all(scores_demo[j] >= scores_demo[j + 1] for j in range(len(scores_demo) - 1)), \
    "los scores deben estar ordenados de mayor a menor"
print("\nDemo de recuperar(pregunta, k=3) — consulta: idea_buscada de T1")
for s, i in top_demo:
    print(f"  score={s:.6f}  idx_chunk={i}  :: {chunks[i][:70]}...")

# ----------------------------------------------------------------------
# 5. Guardar todas las cifras en resultados.json
# ----------------------------------------------------------------------
resultados = {
    "motor_retrieval": MOTOR,
    "shape_indice": [int(V.shape[0]), int(V.shape[1])],
    "tope_tokens_modelo": tope,
    "tokens_fragmento_mas_largo": tokens_mas_largo,
    "cabe_entero": cabe_entero,
    "idx_fragmento_mas_largo": mas_largo,
    "caracteres_fragmento_mas_largo": chars_mas_largo,
    "tokens_perdidos_en_silencio": tokens_perdidos,
    "mensaje_techo": mensaje_techo,
    "modelo_embeddings": "paraphrase-multilingual-MiniLM-L12-v2",
    "num_chunks_indexados": int(len(chunks)),
    "origen_chunks": origen_chunks,
    "tamano_chunk_caracteres": tam,
    "overlap_caracteres": ovl,
    "pregunta_demo": pregunta_demo,
    "demo_top3": [{"score": s, "idx_chunk": i} for s, i in top_demo],
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

print("\n=== Cifras principales (guardadas en resultados.json) ===")
print("motor_retrieval:", resultados["motor_retrieval"])
print("shape_indice:", resultados["shape_indice"])
print("tope_tokens_modelo:", resultados["tope_tokens_modelo"])
print("tokens_fragmento_mas_largo:", resultados["tokens_fragmento_mas_largo"])
print("cabe_entero:", resultados["cabe_entero"])
print("tokens_perdidos_en_silencio:", resultados["tokens_perdidos_en_silencio"])

os.chdir(RAIZ)
mostrar_figuras(RAIZ / 'trabajo_nb' / 'T2')

[T1] cargado entradas/T1.json (24 claves)
[chunks] 9 fragmentos | origen: entradas/T1.json (clave 'chunks')
[índice] sentence-transformers no disponible (ModuleNotFoundError: No module named 'sentence_transformers'); uso el fallback TF-IDF.


Retrieval: léxico (TF-IDF fallback) | índice: (9, 169)
Sin sentence-transformers no hay modelo que trunque: el fallback TF-IDF lee el fragmento entero.

Demo de recuperar(pregunta, k=3) — consulta: idea_buscada de T1
  score=0.341799  idx_chunk=1  :: e directo. Los días no disfrutados se transfieren al año siguiente, ha...
  score=0.251851  idx_chunk=0  :: POLÍTICA DE VACACIONES DE NIMBUSSOFT. Todo empleado tiene derecho a 23...
  score=0.176911  idx_chunk=3  :: POLÍTICA DE TELETRABAJO DE NIMBUSSOFT. El personal puede trabajar en r...

=== Cifras principales (guardadas en resultados.json) ===
motor_retrieval: léxico (TF-IDF fallback)
shape_indice: [9, 169]
tope_tokens_modelo: None
tokens_fragmento_mas_largo: None
cabe_entero: None
tokens_perdidos_en_silencio: None


### Motor e índice: resultados medidos

- **Motor de retrieval:** léxico (TF-IDF fallback) — `sentence-transformers` no está disponible.
- **Modelo de embeddings configurado (intentado):** `paraphrase-multilingual-MiniLM-L12-v2`.
- **Índice:** forma **(9, 169)**; **9 chunks indexados**, tomados de la salida de la Parte 1 (clave `chunks`), con tamaño 300 y overlap 80.

### El techo del modelo, medido

- Fragmento más largo: **300 caracteres**.
- `tope_tokens_modelo`, `tokens_fragmento_mas_largo` y `tokens_perdidos_en_silencio`: **no medidos** — la rama densa no llegó a ejecutarse. El notebook lo dice explícitamente: «Sin sentence-transformers no hay modelo que trunque: el fallback TF-IDF lee el fragmento entero.»
- Consecuencia: con el fallback no hay truncado en silencio que vigilar, pero el techo en tokens del MiniLM queda sin medir en esta ejecución. El material del curso recuerda la asimetría de unidades: el chunking de la Parte 1 cuenta caracteres y el modelo trunca en tokens (regla del material: 500 tokens «aprox.»; notas teóricas y Lab 02: 512; tope del MiniLM del notebook: 128; bge-m3: 8192). Comparar exige convertir, y convertir exige medir.

### Ejercicio 2.1 — `recuperar(pregunta, k=3)`

Demo con la pregunta «hasta un máximo de 5 días» (la idea de la Parte 1). Top-3 medido:

| score | idx_chunk | inicio del chunk |
|---|---|---|
| 0.3418 | 1 | «e directo. Los días no disfrutados se transfieren al año siguiente…» |
| 0.2519 | 0 | «POLÍTICA DE VACACIONES DE NIMBUSSOFT. Todo empleado tiene derecho a 23…» |
| 0.1769 | 3 | «POLÍTICA DE TELETRABAJO DE NIMBUSSOFT. El personal puede trabajar en r…» |

El chunk 1 —el que contiene la idea completa gracias al overlap— encabeza el ranking; el chunk 0, que la corta, queda segundo. El retrieval hereda lo que el corte hizo bien o mal.

## Parte 3 — Generación con contexto

### Estado: no ejecutada — presupuesto agotado

La subtarea T3 **no se completó**. En consecuencia, **esta parte no tiene cifras medidas** y no se reporta ninguna:

- Las **tres pruebas de fuego** no se ejecutaron: la pregunta con respuesta clara («¿puedo trabajar desde el exterior?»), la pregunta sin respuesta en el corpus («¿puedo llevar a mi mascota a la oficina?») y la pregunta ambigua entre documentos («¿qué necesita aprobación del gerente?»).
- No se registró qué chunks recuperó cada prueba ni si la respuesta se apoya en ellos.
- No se verificó la abstención (`se_abstuvo`) en la prueba 2, cuyo mejor comportamiento posible es justamente la frase «El corpus no contiene información suficiente.»: un RAG que siempre responde algo es un generador de alucinaciones con citas decorativas.

**Diseño que la parte debía ejercitar (según el enunciado, sin resultados):** prompt con contexto delimitado entre `<contexto>` y `</contexto>`, instrucción de responder SOLO con base en él, permiso explícito de decir «no sé» y cita del fragmento [n]; generador en cascada (API de Anthropic → H200 de la USFQ vía vLLM → Ollama local → modo inspección con respuestas pre-capturadas); detección de abstención comparando texto normalizado (minúsculas, sin tildes, sin puntuación), no por igualdad de cadena.

## Diagnóstico final

De las cuatro cajas del pipeline (ingesta / chunking / retrieval / generación), la más frágil en este proyecto es el **chunking**. Tres razones, dos de ellas medidas aquí:

1. **El fragmento es la unidad de todo el sistema**: se embebe, se indexa con su identificador, se recupera y se cita en el prompt, y se compara con el golden set. Cambiar el corte cambia los cuatro papeles a la vez, y solo el primero se nota.
2. **Evidencia medida (Parte 1):** con overlap=0, la idea «hasta un máximo de 5 días» no aparece completa en ningún chunk. Ninguna búsqueda repara un mal corte: el retrieval no puede devolver una frase que ningún fragmento contiene entera.
3. **El techo está en tokens y el chunking cuenta caracteres.** Los 300 caracteres de esta Parte 1 son una unidad distinta del tope del modelo, y en esta ejecución ni siquiera pudo medirse ese tope (Parte 2) porque el modelo no estaba disponible: la fragilidad es silenciosa por diseño.

Para vigilarla: (a) contar los tokens del fragmento más largo con el tokenizador del mismo modelo y compararlos con su tope **antes de indexar** — la conducta del Lab 02, que fragmenta en tokens de su propio modelo (512, sin pasar del tope − 2, con solape de un quinto) y avisa si algo no cabe; (b) inspeccionar los cortes con una idea canary, como se hizo aquí con «hasta un máximo de 5 días», exigiendo que aparezca completa en algún chunk; (c) medir retrieval con golden set y métricas (Hit Rate, MRR, Recall@k) para distinguir, ante una falla, si el fragmento correcto no llegó al top-k (recuperación) o llegó y el modelo lo ignoró (generación).

## Cierre

El notebook deja armado un RAG completo en su línea: chunking con overlap (9 chunks de 300 caracteres, overlap 80, verificado con una idea canary), índice y retrieval top-k (fallback TF-IDF, forma (9, 169), demo con top-3 medido) y el diseño de generación con contexto delimitado y derecho a decir «no sé».

**Pendiente declarado:** la Parte 3 (generación y las tres pruebas de fuego) no se ejecutó por presupuesto agotado; sus cifras —chunks recuperados por prueba, apoyo de las respuestas y detección de abstención— quedan sin medir.

Mañana: ¿este RAG es bueno? Métricas de retrieval (Hit Rate, MRR, Recall@k), golden sets, y las extensiones que compiten contra este baseline.